# Isolated QAIRT 2.45 Supertonic artifact acquisition

Acquire and verify the four existing SM8850 / HTP-v81 Supertonic context binaries plus their exact QAIRT `2.45.0.260326154327` Android runtime. Durable artifacts and checkpoints live in Google Drive. The AI Hub credential profile exists only under `/content` for the active Colab session, is never printed, and is deleted in a `finally` block. This notebook does not build an APK, call ADB, touch a phone, or prove phone/NPU execution.

Use a Linux x86_64 Colab runtime. Reconnects are safe: rerun the notebook with the same three hash-pinned controls. Contexts are freshly authenticated and downloaded on every acquisition run because their hashes are not pre-pinned yet; the large runtime is reusable only after its repository-pinned ZIP/member contract passes. The notebook computes the final lock hash immediately after acquire and supplies that in-memory value to standalone verify.


In [ ]:
from pathlib import Path
import atexit
import configparser
import getpass
import hashlib
import json
import os
import platform
import shutil
import subprocess
import sys

EXPECTED = {
    'qairt245_supertonic_hub_acquire.py': '2b7b072989c05de40911d20b1c0ca9764c3a9a16a1eada7bfb06d1d5df2292df',
    'qairt245_supertonic_isolated_spec.json': '79cf01d8bbc9ba783d41196edce0f830a49c1d3e50549a222fdf09aacb3cf8ce',
    'qairt245_supertonic_hub_state.json': '8e6be47a6da10efd2dc1ae667dba3c54119c070cff6fb30ae80c091a6eca71c6',
}
EXPECTED_LANE = 'qairt245_supertonic_isolated_sm8850_htp_v81_v1'
EXPECTED_BUILD = '2.45.0.260326154327'
EXPECTED_HISTORICAL_SPEC_SHA256 = '5ae0e07179362d1c8b3d9500948c7fac05dfaae61c1e7f755f0ad3db2eff655b'

def sha256_bytes(value):
    return hashlib.sha256(value).hexdigest()

def sha256_file(path):
    digest = hashlib.sha256()
    with Path(path).open('rb') as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b''):
            digest.update(block)
    return digest.hexdigest()

if platform.system() != 'Linux' or platform.machine() != 'x86_64':
    raise RuntimeError('Fail-closed: acquisition requires Linux/x86_64')
print({'system': platform.system(), 'machine': platform.machine(), 'python': platform.python_version()})


In [ ]:
from google.colab import drive, files

drive.mount('/content/drive')
DRIVE_ROOT = Path('/content/drive/MyDrive/lmf_qairt245_supertonic_isolated_v1')
CONTROL = DRIVE_ROOT / 'control'
WORKSPACE = DRIVE_ROOT / 'workspace'
SESSION_ROOT = Path('/content/qairt245_supertonic_hub_session')
for directory in (DRIVE_ROOT, CONTROL, WORKSPACE):
    if directory.is_symlink():
        raise RuntimeError(f'Fail-closed: persistent path is a symlink: {directory}')
CONTROL.mkdir(parents=True, exist_ok=True)
WORKSPACE.mkdir(parents=True, exist_ok=True)

for name, expected_hash in EXPECTED.items():
    existing = CONTROL / name
    if existing.is_symlink():
        raise RuntimeError(f'Fail-closed: persistent control is a symlink: {name}')
    if existing.exists() and sha256_file(existing) != expected_hash:
        raise RuntimeError(f'Fail-closed: persistent control hash drift: {name}')
missing = [name for name in EXPECTED if not (CONTROL / name).is_file()]
if missing:
    print('Upload exactly these repository files:', missing)
    uploaded = files.upload()
    for name in missing:
        data = uploaded.get(name)
        if data is None or sha256_bytes(data) != EXPECTED[name]:
            raise RuntimeError(f'Fail-closed: uploaded control mismatch: {name}')
        temporary = CONTROL / f'.{name}.partial'
        temporary_fd = os.open(
            temporary, os.O_WRONLY | os.O_CREAT | os.O_EXCL, 0o600,
        )
        with os.fdopen(temporary_fd, 'wb') as stream:
            stream.write(data)
            stream.flush()
            os.fsync(stream.fileno())
        temporary.replace(CONTROL / name)
for name, expected_hash in EXPECTED.items():
    if sha256_file(CONTROL / name) != expected_hash:
        raise RuntimeError(f'Fail-closed: final control mismatch: {name}')

historical = json.loads((CONTROL / 'qairt245_supertonic_hub_state.json').read_text(encoding='utf-8'))
if historical.get('lane_id') != EXPECTED_LANE or historical.get('spec_sha256') != EXPECTED_HISTORICAL_SPEC_SHA256:
    raise RuntimeError('Fail-closed: historical Hub state identity drift')
if historical.get('status') != 'metadata_authenticated_artifacts_not_downloaded':
    raise RuntimeError('Fail-closed: historical Hub state claim drift')
if historical.get('hub', {}).get('framework', {}).get('full_version') != EXPECTED_BUILD:
    raise RuntimeError('Fail-closed: historical QAIRT build drift')
print({'control_verified': True, 'workspace': str(WORKSPACE), 'session_root': str(SESSION_ROOT)})


Install only the pinned AI Hub client and its declared dependencies into the ephemeral Colab runtime. No QAIRT SDK package, model package, compiler, training stack, or phone tool is installed by this notebook.


In [ ]:
subprocess.run([
    sys.executable, '-m', 'pip', 'install',
    '--disable-pip-version-check', '--no-input', 'qai-hub==0.52.0',
], check=True)
import qai_hub
if qai_hub.__version__ != '0.52.0':
    raise RuntimeError(f'Fail-closed: qai-hub version drift: {qai_hub.__version__}')
print({'qai_hub_version': qai_hub.__version__})


In [ ]:
RUNNER = CONTROL / 'qairt245_supertonic_hub_acquire.py'
SPEC = CONTROL / 'qairt245_supertonic_isolated_spec.json'
subprocess.run([sys.executable, str(RUNNER), 'plan', '--spec', str(SPEC)], check=True)


The next cell prompts with `getpass`, creates a mode-0600 session profile only under `/content` before writing token bytes, and points child processes to it through `QAIHUB_CLIENT_INI`. It deliberately does not call `qai-hub configure`, because that command prints the saved profile including the token in client version 0.52.0. Controller output is captured to redacted, session-only logs and is never echoed. The profile, fresh metadata snapshot, and logs are removed even if acquisition or verification fails. Durable Drive state contains only sanitized artifact identities and hashes.


In [ ]:
def cleanup_session_material():
    if SESSION_ROOT.exists():
        shutil.rmtree(SESSION_ROOT)

cleanup_session_material()
SESSION_ROOT.mkdir(parents=True, exist_ok=True)
SESSION_CONFIG = SESSION_ROOT / 'hub-session.ini'
FRESH_METADATA = SESSION_ROOT / 'fresh-metadata.json'
atexit.register(cleanup_session_material)
try:
    SESSION_SECRET = getpass.getpass('Qualcomm AI Hub API token (hidden): ').strip()
    if not SESSION_SECRET or '\n' in SESSION_SECRET or '\r' in SESSION_SECRET:
        raise RuntimeError('Fail-closed: a single-line AI Hub token is required')
    profile = configparser.ConfigParser(interpolation=None)
    profile['api'] = {
        'api_url': 'https://workbench.aihub.qualcomm.com',
        'web_url': 'https://workbench.aihub.qualcomm.com',
        'api_token': SESSION_SECRET,
        'verbose': 'False',
    }
    temporary_config = SESSION_CONFIG.with_suffix('.partial')
    config_fd = os.open(
        temporary_config, os.O_WRONLY | os.O_CREAT | os.O_EXCL, 0o600,
    )
    with os.fdopen(config_fd, 'w', encoding='utf-8') as stream:
        profile.write(stream)
    temporary_config.chmod(0o600)
    temporary_config.replace(SESSION_CONFIG)
    SESSION_CONFIG.chmod(0o600)
    HUB_ENV = dict(os.environ)
    HUB_ENV['QAIHUB_CLIENT_INI'] = str(SESSION_CONFIG)
except BaseException:
    cleanup_session_material()
    SESSION_SECRET = None
    profile = None
    raise

def redact_session_output(value):
    return value.replace(SESSION_SECRET, '[REDACTED]')

def run_private(label, arguments):
    completed = subprocess.run(
        [sys.executable, str(RUNNER), *arguments],
        env=HUB_ENV, text=True, stdout=subprocess.PIPE, stderr=subprocess.PIPE,
        check=False,
    )
    log_path = SESSION_ROOT / f'{label}.redacted.log'
    log_path.write_text(
        redact_session_output(completed.stdout + '\n' + completed.stderr),
        encoding='utf-8',
    )
    log_path.chmod(0o600)
    if completed.returncode:
        raise RuntimeError(
            f'{label} failed closed with code {completed.returncode}; '
            f'details are in the session-only redacted log'
        )

try:
    run_private('metadata', [
        'metadata', '--spec', str(SPEC), '--state', str(FRESH_METADATA),
    ])
    fresh = json.loads(FRESH_METADATA.read_text(encoding='utf-8'))
    print({
        'metadata_status': fresh['status'],
        'lane_id': fresh['lane_id'],
        'qairt_build': fresh['hub']['framework']['full_version'],
        'authenticated_components': list(fresh['hub']['components']),
        'runtime_central_directory_sha256': fresh['runtime']['central_directory']['sha256'],
    })
    run_private('acquire', [
        'acquire', '--spec', str(SPEC), '--workspace', str(WORKSPACE),
    ])
    lock_path = WORKSPACE / 'qairt245_supertonic_artifact_lock.json'
    TRUSTED_LOCK_SHA256 = sha256_file(lock_path)
    run_private('verify', [
        'verify', '--spec', str(SPEC), '--workspace', str(WORKSPACE),
        '--expected-lock-sha256', TRUSTED_LOCK_SHA256,
    ])
    lock = json.loads(lock_path.read_text(encoding='utf-8'))
    if lock['status'] != 'artifacts_locked_unpackaged_unexecuted':
        raise RuntimeError('Fail-closed: unexpected artifact-lock claim')
    if lock['apk_built'] or lock['phone_execution_proven'] or lock['npu_execution_proven']:
        raise RuntimeError('Fail-closed: artifact lock overclaims execution')
    summary = {
        'status': lock['status'],
        'artifact_lock': str(lock_path),
        'artifact_lock_sha256': TRUSTED_LOCK_SHA256,
        'controller_sha256': lock['controller_sha256'],
        'metadata_fingerprint': lock['metadata_fingerprint'],
        'context_sha256': {
            name: record['file']['sha256']
            for name, record in lock['contexts'].items()
        },
        'runtime_archive_sha256': lock['runtime_archive']['file']['sha256'],
        'runtime_member_sha256': {
            record['path']: record['sha256'] for record in lock['runtime_members']
        },
        'apk_built': False,
        'phone_execution_proven': False,
        'npu_execution_proven': False,
    }
    print(json.dumps(summary, indent=2, sort_keys=True))
finally:
    cleanup_session_material()
    SESSION_SECRET = None
    profile = None
    HUB_ENV = None
    TRUSTED_LOCK_SHA256 = None
    print({'session_profile_removed': not SESSION_ROOT.exists()})
